# Privacidade e Ética dos Dados

Aqui, avaliaremos se os dados processados, mesmo sem nome ou CPF, permitem reidentificar pacientes, e quais grupos ficam sub-representados.

In [1]:
import pandas as pd

df = pd.read_csv('../dados/processados/internacoes_icsap_pr.csv.gz', sep=';',
                 dtype={'N_AIH': str, 'MUNIC_RES': str, 'DT_INTER': str, 'DIAG_PRINC': str})
print('Registros:', len(df))

Registros: 662730


In [2]:
print('Colunas:', list(df.columns))
print('N_AIH unicos:', df['N_AIH'].nunique(), 'de', len(df))

Colunas: ['N_AIH', 'MUNIC_RES', 'DT_INTER', 'DIAG_PRINC', 'grupo_icsap', 'VAL_TOT', 'DIAS_PERM', 'flag_dias_zero', 'flag_dias_alto', 'flag_valor_zero']
N_AIH unicos: 662730 de 662730


A base não tem nome, CPF nem endereço, mas cada linha tem um N_AIH único (662.730 de 662.730). Ele identifica a internação e, com ela, uma pessoa específica. Quem tiver acesso ao número da AIH, como o hospital ou o próprio paciente, encontra o registro.

In [3]:
grupos = df.groupby(['MUNIC_RES', 'DT_INTER', 'DIAG_PRINC']).size()
unicas = (grupos == 1).sum()
print('Combinacoes municipio + data + CID:', len(grupos))
print('Combinacoes com uma so internacao:', unicas)
print('% das internacoes que sao unicas:', round(unicas / len(df) * 100, 2))

Combinacoes municipio + data + CID: 556326
Combinacoes com uma so internacao: 491608
% das internacoes que sao unicas: 74.18


74,2% das internações (491.608) são únicas na combinação município + data de internação + CID. Ou seja, quem sabe que alguém de uma cidade foi internado num certo dia consegue, na maioria dos casos, achar o registro e descobrir o diagnóstico, o valor pago e os dias de internação. Mesmo sem nome, a base é **reidentificável**.

In [4]:
df['ano'] = df['DT_INTER'].str[:4]
contagem = df.groupby(['MUNIC_RES', 'ano', 'grupo_icsap']).size()
print('Combinacoes municipio + ano + grupo:', len(contagem))
print('Com menos de 5 casos:', (contagem < 5).sum())

Combinacoes municipio + ano + grupo: 29669
Com menos de 5 casos: 12247


Das 29.669 combinações de município x ano x grupo ICSAP, 12.247 (41%) têm menos de 5 casos. Uma tabela publicada nesse nível de detalhe exporia pessoas, então essas células precisam ser suprimidas ou agregadas (por exemplo, por região de saúde ou somando grupos).

In [5]:
por_municipio = df.groupby('MUNIC_RES').size().sort_values()
print('Municipios com menos de 100 internacoes no periodo:', (por_municipio < 100).sum())
print(por_municipio.head(10))

Municipios com menos de 100 internacoes no periodo: 4
MUNIC_RES
411260     62
411650     64
411160     88
412830     91
411435    103
410315    107
411830    109
410715    115
411535    125
411585    128
dtype: int64


4 municípios têm menos de 100 internações ICSAP em 4,5 anos (o menor tem 62, uns 14 por ano). Neles, a taxa é instável, já que poucos casos a mais ou a menos mudam muito o resultado. São grupos sub-representados, e qualquer comparação precisa levar isso em conta.